In [3]:
import sys

print(sys.executable)

c:\Users\risha\OneDrive\Documents\Algoverse\.venv\Scripts\python.exe


In [4]:
import sys
import torchxrayvision as xrv

print("Python:", sys.executable)

NIH_FINDINGS = [
    "Atelectasis",
    "Consolidation",
    "Infiltration",
    "Pneumothorax",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Effusion",
    "Pneumonia",
    "Pleural_Thickening",
    "Cardiomegaly",
    "Nodule",
    "Mass",
    "Hernia",
]

nih_model = xrv.models.DenseNet(weights="densenet121-res224-nih")
all_model = xrv.models.DenseNet(weights="densenet121-res224-all")

nih_model.eval()
all_model.eval()

nih_labels = list(nih_model.targets)
all_labels = list(all_model.targets)

print("NIH labels:", nih_labels)
print("All labels:", all_labels)

assert nih_labels[:14] == NIH_FINDINGS
assert nih_labels[14:] == ["", "", "", ""]
assert all_labels[:14] == NIH_FINDINGS

print("PASS: models loaded and label ordering is correct.")

Python: c:\Users\risha\OneDrive\Documents\Algoverse\.venv\Scripts\python.exe
NIH labels: ['Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia', 'Pleural_Thickening', 'Cardiomegaly', 'Nodule', 'Mass', 'Hernia', '', '', '', '']
All labels: ['Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia', 'Pleural_Thickening', 'Cardiomegaly', 'Nodule', 'Mass', 'Hernia', 'Lung Lesion', 'Fracture', 'Lung Opacity', 'Enlarged Cardiomediastinum']
PASS: models loaded and label ordering is correct.


## Second backbone: ResNet-50 schema check

The Week 3 downstream contract remains the ordered 18-label schema from `densenet121-res224-all`. This cell loads TorchXRayVision's `resnet50-res512-all`, runs one normalized dummy image through it, and stops if either the output shape or label ordering differs.

In [5]:
import torch

REFERENCE_SCHEMA = list(all_model.targets)

resnet_model = xrv.models.ResNet(weights="resnet50-res512-all")
resnet_model.eval()
resnet_labels = list(resnet_model.targets)

# TorchXRayVision will resize this 224x224 input to ResNet-50's native 512x512 resolution.
dummy_image = torch.linspace(-1024, 1024, 224 * 224, dtype=torch.float32).reshape(1, 1, 224, 224)
with torch.inference_mode():
    resnet_output = resnet_model(dummy_image)

print("ResNet-50 labels:", resnet_labels)
print("ResNet-50 output shape:", tuple(resnet_output.shape))

assert resnet_labels == REFERENCE_SCHEMA, (
    f"ResNet-50 schema mismatch!\nExpected: {REFERENCE_SCHEMA}\nActual: {resnet_labels}"
)
assert tuple(resnet_output.shape) == (1, len(REFERENCE_SCHEMA))
assert all(resnet_labels)

print("PASS: ResNet-50 uses the identical ordered 18-label output schema.")

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/pc-nih-rsna-siim-vin-resnet50-test512-e400-state.pt -O C:\Users\risha\.torchxrayvision\models_data/pc-nih-rsna-siim-vin-resnet50-test512-e400-state.pt`
[██████████████████████████████████████████████████]
ResNet-50 labels: ['Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia', 'Pleural_Thickening', 'Cardiomegaly', 'Nodule', 'Mass', 'Hernia', 'Lung Lesion', 'Fracture', 'Lung Opacity', 'Enlarged Cardiomediastinum']
ResNet-50 output shape: (1, 18)
PASS: ResNet-50 uses the identical ordered 18-label output schema.


## Week 3 model selection and references

Use only the two models below in the strict Week 3 comparison because they expose the same ordered 18-label output schema. The other public candidates are recorded as adapter candidates, not silently mixed into the main pipeline.

### Strict drop-in models

- **DenseNet-121:** `densenet121-res224-all` — [TorchXRayVision repository](https://github.com/mlmed/torchxrayvision), [model card](https://huggingface.co/torchxrayvision/densenet121-res224-all)
- **ResNet-50:** `resnet50-res512-all` — [TorchXRayVision repository](https://github.com/mlmed/torchxrayvision)

### Public adapter candidates

- `densenet121-res224-nih`: 14 valid NIH findings plus 4 blank/untrained slots.
- `densenet121-res224-chex`: CheXpert-specific trained-label subset.
- `densenet121-res224-mimic_ch`: MIMIC-CXR-specific trained-label subset.
- [CheXzero](https://github.com/rajpurkarlab/CheXzero): prompt-dependent zero-shot scores rather than a fixed 18-label head.
- [CXR Foundation](https://github.com/Google-Health/cxr-foundation): chest-X-ray embeddings rather than fixed pathology scores.
- [Ark/Ark+](https://github.com/JLiangLab/Ark): pretrained foundation encoder; it does not expose the same 18-label classifier head.

### DenseNet references

- [TorchXRayVision DenseNet implementation](https://github.com/mlmed/torchxrayvision/blob/main/torchxrayvision/models.py)
- [Original DenseNet repository](https://github.com/liuzhuang13/DenseNet)
- [Densely Connected Convolutional Networks paper](https://arxiv.org/abs/1608.06993)

In [6]:
# Reusable strict registry for the Week 3 downstream pipeline.
STRICT_MODEL_REGISTRY = [
    {
        "name": "densenet121-res224-all",
        "model": all_model,
        "native_resolution": 224,
        "github": "https://github.com/mlmed/torchxrayvision",
    },
    {
        "name": "resnet50-res512-all",
        "model": resnet_model,
        "native_resolution": 512,
        "github": "https://github.com/mlmed/torchxrayvision",
    },
]

for entry in STRICT_MODEL_REGISTRY:
    model = entry["model"]
    labels = list(model.targets)
    assert labels == REFERENCE_SCHEMA, f"Schema mismatch for {entry['name']}"
    assert all(labels), f"Blank label found in {entry['name']}"
    print(f"PASS: {entry['name']} | resolution={entry['native_resolution']} | outputs={len(labels)}")

ADAPTER_CANDIDATES = [
    {"name": "densenet121-res224-nih", "status": "adapter_required", "reason": "14 valid NIH labels plus 4 blank/untrained slots"},
    {"name": "densenet121-res224-chex", "status": "adapter_required", "reason": "CheXpert-specific trained-label subset"},
    {"name": "densenet121-res224-mimic_ch", "status": "adapter_required", "reason": "MIMIC-CXR-specific trained-label subset"},
]
print("\nOptional candidates are documented but excluded from the strict schema:")
for candidate in ADAPTER_CANDIDATES:
    print(f"- {candidate['name']}: {candidate['status']} ({candidate['reason']})")

print("\nPASS: strict registry is safe for identical downstream code.")

PASS: densenet121-res224-all | resolution=224 | outputs=18
PASS: resnet50-res512-all | resolution=512 | outputs=18

Optional candidates are documented but excluded from the strict schema:
- densenet121-res224-nih: adapter_required (14 valid NIH labels plus 4 blank/untrained slots)
- densenet121-res224-chex: adapter_required (CheXpert-specific trained-label subset)
- densenet121-res224-mimic_ch: adapter_required (MIMIC-CXR-specific trained-label subset)

PASS: strict registry is safe for identical downstream code.


## Works-cited starter entries

```bibtex
@inproceedings{cohen2022torchxrayvision,
  title={TorchXRayVision: A library of chest X-ray datasets and models},
  author={Cohen, Joseph Paul and Viviano, Joseph D. and Bertin, Paul and others},
  booktitle={Medical Imaging with Deep Learning},
  year={2022},
  url={https://github.com/mlmed/torchxrayvision}
}

@inproceedings{huang2017densely,
  title={Densely Connected Convolutional Networks},
  author={Huang, Gao and Liu, Zhuang and van der Maaten, Laurens and Weinberger, Kilian Q.},
  booktitle={CVPR},
  year={2017},
  url={https://github.com/liuzhuang13/DenseNet}
}

@article{tiu2022chexzero,
  title={Expert-level detection of pathologies from unannotated chest X-ray images via self-supervised learning},
  author={Tiu, Ekin and Talius, Ellie and Patel, Pujan and others},
  journal={Nature Biomedical Engineering},
  year={2022},
  doi={10.1038/s41551-022-00936-9},
  url={https://github.com/rajpurkarlab/CheXzero}
}

@article{ma2025ark,
  title={A fully open AI foundation model applied to chest radiography},
  author={Ma, DongAo and Pang, Jiaxuan and Gotway, Michael B. and Liang, Jianming},
  journal={Nature},
  year={2025},
  url={https://github.com/JLiangLab/Ark}
}
```